# AgileTaskHeron

**Automated Agile Task Force Optimization using Heterogeneous Graphs**

This notebook demonstrates a powerful approach to automatically group and prioritize tasks in an Agile environment.

By modeling **Tasks**, **Required Skills**, and **Human Resources** (with their past experiences) as a graph, we can:

- Identify the most critical tasks using **PageRank**
- Discover natural **Task-Skill-Person bundles** using **Louvain community detection**
- Generate ready-to-use **Agile sprint priority + recommended team bundles**

---

### Let's build a graph that captures:
- Task priorities and dependencies
- Required skill sets for each task
- Available human resources and their relevant experience

All in one unified graph structure.

## Table of Contents
1. [Setup & Imports](#setup)
2. [Build the Graph and Run Analysis](#build-graph)
3. [View Agile Priority Bundles](#view-bundles)

## Why Graph + PageRank + Louvain?

Traditional Agile tools treat tasks and people separately.  
This notebook connects them in a **heterogeneous graph** so that:

- Tasks with many dependencies or critical skills naturally rise to the top (PageRank)
- Related tasks, skills, and people automatically form meaningful **bundles** (Louvain)
- You get both **priority order** and **ready-to-assign teams** in one run

This is especially powerful in large organizations where AI/automation and human resources must be balanced.

## Edge Weights Explained

In this graph, every relationship (edge) has a **`weight`** value that represents **how important or strong** that connection is.

### Important Rules for `weight`:
- Must be **greater than 0** (0 is **not allowed**)
- Range: **0.1 ~ 1.0** (minimum 0.1, maximum 1.0)
- Higher value = **more important** from the HR / Agile team's perspective

### How to decide the weight?
- This is **subjective** — it reflects your team's real-world judgment.
- Example:
  - `Person1` → `Assigned_To` → `Task1` with **weight = 1.0**  
    → "This person is critically important for this task"
  - `Person3` → `Completed` → `Task1` with **weight = 0.2**  
    → "This person has some experience, but not very critical"

**Tip for HR teams**:  
When you create the first `edges` DataFrame, carefully assign weights based on **business priority**, **past performance**, and **strategic importance**.  
The algorithm will use these weights to calculate task priority (PageRank) and natural bundles (Louvain).

---

Now let's create the edge data:
<a id="setup"></a>

In [1]:
import cudf
import cugraph

edges = cudf.DataFrame({
	    'src': ['Person1',       'Person2',      'Task1',     'Person1',    'Task1',         'Task2',          'Person2',   'Person3', 'Person3' ],
        'dst': ['Task1',          'Task2',         'Task2',    'Skill1',     'Skill1',       'Skill2',         'Skill2',    'Task1',   'Skill1' ],
  'edge_type': ['Assigned_To',   'Completed',   'Depends_On', 'Has_Skill', 'Requires_Skill', 'Requires_Skill', 'Has_Skill', 'Completed',   'Has_Skill' ],
  'weight': [1.0, 0.8, 0.5, 0.7, 0.6, 0.6, 0.6, 0.2, 0.2 ]

})

## Step 2: Build the Graph and Run Analysis

Now that we have defined the edges with proper weights, let's:

1. Create a **cuGraph** (GPU-accelerated graph)
2. Run **PageRank** to calculate task importance
3. Run **Louvain community detection** to discover natural Task-Skill-Person bundles
4. Generate the final **Agile priority bundles**

All computation happens on GPU for speed.
<a id="build-graph"></a>

In [2]:
G = cugraph.Graph()
try:
	G.from_cudf_edgelist(edges, source='src', destination='dst', edge_attr='weight', store_transposed=True)
	print ("MultiGraph Generation Successful")
except Exception as e:
	print (f"MultiGraph Generation Failed: {e}")

print(f"✅ Graph created successfully!")
print(f"   - Nodes: {G.number_of_nodes()}")
print(f"   - Edges: {G.number_of_edges()}")

# Run PageRank (task importance)
pagerank_scores = cugraph.pagerank(G)

# Run Louvain (natural bundles)
workgroup, modularity = cugraph.louvain(G)
workgroup = workgroup.merge(pagerank_scores, on='vertex', how='left')

print(f"✅ Louvain community detection completed (modularity: {modularity:.4f})")

MultiGraph Generation Successful
✅ Graph created successfully!
   - Nodes: 7
   - Edges: 9
✅ Louvain community detection completed (modularity: 0.3948)


### If we print pagerank_scores, 

In [3]:
pagerank_scores

,pagerank,vertex
0,0.212795,Task1
1,0.176054,Task2
2,0.146901,Skill1
3,0.158341,Person1
4,0.117660,Skill2
5,0.053806,Person3
6,0.134442,Person2


## Step 3: View Agile Priority Bundles
<a id="view-bundles"></a>

In [6]:
# 3. View Agile Priority Bundles
print("=== 🔥 Agile Task Priority Bundles ===\n")

partition_labels = workgroup['partition'].unique().to_arrow().to_pylist()

for part in sorted(partition_labels):
    group = workgroup[workgroup['partition'] == part]
    
    tasks = group[group['vertex'].str.startswith('Task')]['vertex'].to_pandas().tolist()
    if not tasks:
        continue
    task_name = tasks[0]
    
    skills = group[group['vertex'].str.startswith('Skill')]['vertex'].to_pandas().tolist()
    people = group[group['vertex'].str.startswith('Person')]
    
    task_score = float(group[group['vertex'] == task_name]['pagerank'].iloc[0])
    
    print(f"{part+1} Ranking ─ {task_name} (importance: {task_score:.4f})")
    print(f"   • Required Skills     : {skills}")
    print(f"   • Total Nodes in Bundle : {len(group)}")
    print(f"   • Nodes               : {group['vertex'].to_pandas().tolist()}")
    print(f"   • Human Resources     :")
    
    people_pd = people.to_pandas()
    for _, person_row in people_pd.iterrows():
        person_name = person_row['vertex']
        person_score = float(person_row['pagerank'])
        print(f"       - {person_name:<10} (contribution: {person_score:.4f})")
    
    print("-" * 70)

=== 🔥 Agile Task Priority Bundles ===

1 Ranking ─ Task1 (importance: 0.2128)
   • Required Skills     : ['Skill1']
   • Total Nodes in Bundle : 4
   • Nodes               : ['Task1', 'Skill1', 'Person1', 'Person3']
   • Human Resources     :
       - Person1    (contribution: 0.1583)
       - Person3    (contribution: 0.0538)
----------------------------------------------------------------------
2 Ranking ─ Task2 (importance: 0.1761)
   • Required Skills     : ['Skill2']
   • Total Nodes in Bundle : 3
   • Nodes               : ['Task2', 'Skill2', 'Person2']
   • Human Resources     :
       - Person2    (contribution: 0.1344)
----------------------------------------------------------------------
